<a href="https://colab.research.google.com/github/AvirupVIP/SIGNIFY---Signature-Forgery-Detection-System/blob/master/MobileNet_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle
!pip install opencv-python
!pip install scikit-learn

In [2]:
import os
import json

kaggle_config = {
  "username":"sandandipto",
  "key":"KGAT_d5a3d64f806d070d376a3862e1c6f371"
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_config, f)

!chmod 600 /root/.kaggle/kaggle.json

In [3]:
!kaggle datasets download -d shreelakshmigp/cedardataset

Dataset URL: https://www.kaggle.com/datasets/shreelakshmigp/cedardataset
License(s): unknown
cedardataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
!unzip -q cedardataset.zip


replace signatures/Readme.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [5]:
!pip install opencv-python scikit-learn tqdm

In [6]:
import os
import cv2
import numpy as np
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split

In [7]:
genuine_path = "signatures/full_org"
forged_path = "signatures/full_forg"

In [8]:
IMG_SIZE = 224

def preprocess_signature(path):

    img = cv2.imread(path)

    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(
        gray,0,255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = np.ones((3,3),np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    coords = cv2.findNonZero(opening)

    if coords is None:
        return None

    x,y,w,h = cv2.boundingRect(coords)
    crop = opening[y:y+h, x:x+w]

    resized = cv2.resize(crop,(IMG_SIZE,IMG_SIZE))

    normalized = resized/255.0

    normalized = np.stack((normalized,)*3, axis=-1)

    return normalized

In [9]:
images = []
labels = []

print("Loading Genuine Signatures")

for img in tqdm(os.listdir(genuine_path)):

    if img.endswith(".png"):

        path = os.path.join(genuine_path,img)

        data = preprocess_signature(path)

        if data is not None:
            images.append(data)
            labels.append(1)


print("Loading Forged Signatures")

for img in tqdm(os.listdir(forged_path)):

    if img.endswith(".png"):

        path = os.path.join(forged_path,img)

        data = preprocess_signature(path)

        if data is not None:
            images.append(data)
            labels.append(0)


X = np.array(images)
y = np.array(labels)

print(X.shape,y.shape)

Loading Genuine Signatures


100%|██████████| 1321/1321 [00:11<00:00, 117.39it/s]


Loading Forged Signatures


100%|██████████| 1321/1321 [00:05<00:00, 220.91it/s]


(2640, 224, 224, 3) (2640,)


In [10]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
     X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256,activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(1,activation='sigmoid')(x)

model = Model(inputs=base_model.input,outputs=output)

In [12]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=10,
    batch_size=32
)

In [ ]:
def predict_signature(path):

    img = preprocess_signature(path)

    if img is None:
        print("Invalid image")
        return

    img = np.expand_dims(img,axis=0)

    prob = model.predict(img)[0][0]

    if prob > 0.5:
        print("Prediction: Genuine")
    else:
        print("Prediction: Forged")

    print("Confidence:",prob)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
predict_signature("Forge_Sig.jpeg")